# EDA: GroupKFold fold-count sensitivity

Exploration only, nothing here is load-bearing on its own. Supports the `N_SPLITS` choice in `src/scoring.py`.

**Question:** `N_SPLITS=7` was picked as a middle ground between 5 (cheap, standard) and 10 (more stable, costlier) *before any model existed to test it* -- a compute/variance tradeoff, not a signal-shape property, decided that way on purpose. Now that XGBoost exists for all four datasets, does the actual choice of k materially change what GroupKFold CV reports (mean RMSE, spread across folds)? If not, 7 is confirmed as good as any reasonable alternative and the upcoming FD002/FD004 tuning pass can trust CV RMSE differences as real. If it does move meaningfully, that's a decision to revisit before tuning, not just a footnote.

In [1]:
import os
import sys

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd

from src.features import add_rolling_features, add_rul_targets, add_savgol_features, find_constant_sensors
from src.load import DATASETS, SENSOR_COLS, load_train
from src.models import build_feature_cols, make_xgb_model
from src.regimes import fit_regimes
from src.scoring import group_kfold_cv


def build_train_features(dataset: str):
    train = load_train(dataset)
    train_norm, _ = fit_regimes(train)
    constant_sensors = find_constant_sensors(train_norm)
    varying_sensors = [s for s in SENSOR_COLS if s not in constant_sensors]
    train_feat = add_rolling_features(train_norm, varying_sensors)
    train_feat = add_savgol_features(train_feat, varying_sensors)
    train_feat = add_rul_targets(train_feat)
    feature_cols = build_feature_cols(varying_sensors)
    return train_feat, feature_cols

In [2]:
K_VALUES = [5, 7, 10]

rows = []
for dataset in DATASETS:
    train_feat, feature_cols = build_train_features(dataset)
    for k in K_VALUES:
        fold_rmses = group_kfold_cv(make_xgb_model, train_feat, feature_cols, "rul_capped", n_splits=k)
        rows.append({
            "dataset": dataset,
            "k": k,
            "mean_rmse": np.mean(fold_rmses),
            "std_rmse": np.std(fold_rmses),
            "min_rmse": np.min(fold_rmses),
            "max_rmse": np.max(fold_rmses),
        })

results = pd.DataFrame(rows)
results

,dataset,k,mean_rmse,std_rmse,min_rmse,max_rmse
0,FD001,5,15.330602,0.507095,14.512011,16.086923
1,FD001,7,15.783351,1.629450,13.719974,18.569079
2,FD001,10,15.281575,1.481130,12.711854,17.117526
3,FD002,5,16.113090,0.648362,15.225116,16.985304
4,FD002,7,15.834899,0.997123,13.990828,17.017343
5,FD002,10,15.789714,1.497717,13.460122,18.788755
6,FD003,5,14.029634,0.961407,12.908447,15.567254
7,FD003,7,13.839781,2.438396,10.577709,19.147422
8,FD003,10,13.336763,1.594927,10.993492,15.544427
9,FD004,5,15.547111,1.292347,13.616798,17.088984


In [3]:
print("mean CV RMSE by dataset x k:")
print(results.pivot(index="dataset", columns="k", values="mean_rmse").round(2))
print()
print("std of fold RMSE by dataset x k (spread across folds, lower = more stable):")
print(results.pivot(index="dataset", columns="k", values="std_rmse").round(2))

mean CV RMSE by dataset x k:
k           5      7      10
dataset                     
FD001    15.33  15.78  15.28
FD002    16.11  15.83  15.79
FD003    14.03  13.84  13.34
FD004    15.55  15.43  15.45

std of fold RMSE by dataset x k (spread across folds, lower = more stable):
k          5     7     10
dataset                  
FD001    0.51  1.63  1.48
FD002    0.65  1.00  1.50
FD003    0.96  2.44  1.59
FD004    1.29  1.54  1.44


**Finding:** `N_SPLITS=7` is confirmed as good as any reasonable alternative. Mean CV RMSE barely moves across k=5/7/10 for any dataset -- the largest swing is FD003 (14.03 / 13.84 / 13.34, a ~0.7 RMSE range), and FD001/FD002/FD004 all move less than 0.5. That's the number that matters for the upcoming tuning pass: a hyperparameter change needs to move mean CV RMSE by more than this natural fold-count wobble (roughly 0.3-0.7 depending on dataset) to be trusted as a real improvement, not noise.

The per-fold spread (`std_rmse`) is a separate story and genuinely dataset-dependent -- FD001 and FD003 both happen to have their *highest* per-fold spread at k=7 specifically (FD003: fold RMSEs ranging 10.58-19.15 at k=7, vs. a tighter 12.91-15.57 at k=5), meaning a handful of folds catch a harder group of engines depending on exactly how GroupKFold's deterministic (non-shuffled) unit assignment falls out at that k. This doesn't argue for a different k -- it would show up at other k values too, just with different folds affected -- but it's a reason to always read CV as the mean across folds, never trust one fold's number alone.

No change to `scoring.py`: `N_SPLITS` stays at 7.